# Step 05 — Traffic Quality Gate & Corridor Probe Planning


In [6]:

import copy
import json
import pickle
import shutil
from datetime import datetime, timezone

import networkx as nx
import pandas as pd
from IPython.display import display

from routing.config import (
    RESULTS_DIR,
    ORIGIN,
    ORIGINS,
    DESTINATION,
)
from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.traffic_api import TomTomClient
from routing.traffic_pipeline import (
    collect_traffic_snapshot,
    make_routing_graph,
    save_summary,
)
from routing.traffic_quality import (
    plan_corridor_probes,
    audit_traffic_snapshot,
)
from routing.probe_history import (
    load_probe_history,
    update_probe_history,
    save_probe_history,
)

RADIUS_M = 2000
G = load_road_graph(radius_m=RADIUS_M)

assert isinstance(G, nx.MultiDiGraph)
assert G.is_directed()

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("========== NOTEBOOK 05 INITIALIZED ==========")
print("Graph:", type(G).__name__)
print("Nodes:", G.number_of_nodes())
print("Directed edges:", G.number_of_edges())
print("Results directory:", RESULTS_DIR)
print("PASS: Imports and OSM graph loaded")


========== NOTEBOOK 05 INITIALIZED ==========
Graph: MultiDiGraph
Nodes: 2569
Directed edges: 5850
Results directory: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results
PASS: Imports and OSM graph loaded


## 1 — Load and protect the existing Snapshot


In [7]:

snapshot_path = RESULTS_DIR / "traffic_snapshot.pkl"

previous_bundle = None
previous_snapshot = None
previous_summary = {}

def _assert_same_topology(base_graph, candidate_graph, label):
    if not isinstance(candidate_graph, nx.MultiDiGraph):
        raise TypeError(f"{label} must be a networkx.MultiDiGraph")

    if set(base_graph.nodes) != set(candidate_graph.nodes):
        raise ValueError(f"{label} node set differs from the current OSM graph")

    if set(base_graph.edges(keys=True)) != set(candidate_graph.edges(keys=True)):
        raise ValueError(f"{label} directed-edge set differs from the current OSM graph")


if snapshot_path.exists():
    backup_path = RESULTS_DIR / (
        "traffic_snapshot_before_merge_"
        + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        + ".pkl"
    )
    shutil.copy2(snapshot_path, backup_path)

    with snapshot_path.open("rb") as f:
        raw_bundle = pickle.load(f)

    # Support both the current bundle format and an older raw-graph file.
    if isinstance(raw_bundle, dict):
        if "snapshot" not in raw_bundle:
            raise ValueError(
                "traffic_snapshot.pkl is a dictionary but has no 'snapshot' key."
            )
        previous_bundle = raw_bundle
        previous_snapshot = raw_bundle["snapshot"]

    elif isinstance(raw_bundle, nx.MultiDiGraph):
        previous_snapshot = raw_bundle
        previous_bundle = {"snapshot": raw_bundle, "summary": {}}

    else:
        raise TypeError(
            "Unsupported traffic_snapshot.pkl format: "
            f"{type(raw_bundle).__name__}"
        )

    _assert_same_topology(G, previous_snapshot, "saved snapshot")
    previous_summary = previous_bundle.get("summary", {})

    print("Existing snapshot loaded:", snapshot_path)
    print("Backup created:", backup_path)
    print("Nodes:", previous_snapshot.number_of_nodes())
    print("Edges:", previous_snapshot.number_of_edges())

else:
    print("No previous snapshot exists. A new one will be created after live collection.")

print("PASS: Snapshot loading/topology validation completed")


Existing snapshot loaded: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot.pkl
Backup created: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot_before_merge_20261108_075220.pkl
Nodes: 2569
Edges: 5850
PASS: Snapshot loading/topology validation completed


## 2 — Plan corridor probes


In [8]:

history = load_probe_history()

REQUEST_LIMIT = 12

plan = plan_corridor_probes(
    G,
    max_probes=REQUEST_LIMIT,
    origins=ORIGINS,
    destination=DESTINATION,
    corridor_width_m=350,
    min_length_m=35,
    min_spacing_m=65,
    history=history,
)

plan_df = pd.DataFrame(plan)

print("Maximum corridor probes:", REQUEST_LIMIT)
print("Planned corridor probes:", len(plan))

if not plan_df.empty and "road_class" in plan_df.columns:
    print("\nRoad-class distribution:")
    display(plan_df["road_class"].value_counts().rename("count").to_frame())

display(plan_df)


Maximum corridor probes: 12
Planned corridor probes: 12

Road-class distribution:


,count
road_class,
trunk,8
primary,4


,edge_id,latitude,longitude,length_m,road_class,corridor,corridor_offset_m
0,"(265565434, 109870, 0)",51.495027,-0.182279,72.25,trunk,Gloucester Road,34.0
1,"(20947178, 1691189798, 0)",51.501582,-0.177494,89.35,primary,Royal Albert Hall,77.2
2,"(6280192964, 3818359076, 0)",51.495869,-0.172541,140.77,trunk,South Kensington,115.2
3,"(265565438, 265565434, 0)",51.495122,-0.181017,103.78,trunk,Gloucester Road,76.0
4,"(385312258, 20947178, 0)",51.501568,-0.178813,93.45,primary,Royal Albert Hall,107.2
5,"(5898995362, 2180741854, 0)",51.495479,-0.176346,180.05,trunk,South Kensington,149.0
6,"(5940891836, 1356901661, 0)",51.494993,-0.183865,143.90,trunk,Gloucester Road,100.8
7,"(109670, 1691189798, 0)",51.501654,-0.175713,158.14,primary,Royal Albert Hall,153.8
8,"(3818359116, 1960925507, 0)",51.496182,-0.170504,185.37,trunk,South Kensington,256.7
9,"(267342172, 265565438, 0)",51.495199,-0.179982,40.51,trunk,Gloucester Road,110.7


## 3 — Select route-focused probes



In [9]:
USE_ROUTE_FOCUSED_PROBES = True
BATCH_SIZE = 6

route_probe_path = RESULTS_DIR / "route_probe_candidates.json"

if USE_ROUTE_FOCUSED_PROBES and route_probe_path.exists():
    candidate_records = json.loads(route_probe_path.read_text(encoding="utf-8"))
    if candidate_records:
        print("Source: Notebook 06 route-focused candidates")
    else:
        candidate_records = plan
        print("Notebook 06 exported no eligible route edges; using the corridor plan.")
elif USE_ROUTE_FOCUSED_PROBES:
    candidate_records = plan
    print("Route candidate file is missing; using the corridor plan instead.")
else:
    candidate_records = plan
    print("Source: corridor plan")

usable_statuses = {"red", "yellow", "clear", "closed"}

def _has_usable_observation(attrs):
    return (
        attrs.get("traffic_matched") is True
        and attrs.get("time_is_imputed") is False
        and attrs.get("traffic_status") in usable_statuses
    )

def _edge_history_attempts(edge_id):
    edge_history = history.get("edges", {}) if isinstance(history, dict) else {}
    item = edge_history.get("|".join(map(str, edge_id)), {})
    try:
        return max(0, int(item.get("attempts", 0)))
    except (AttributeError, TypeError, ValueError):
        return 0

pending_probe_records = []
seen_edge_ids = set()
skipped_observed = 0
skipped_short = 0
skipped_max_attempts = 0

for record in candidate_records:
    edge_id = tuple(record["edge_id"])
    if edge_id in seen_edge_ids:
        continue
    seen_edge_ids.add(edge_id)

    if not G.has_edge(*edge_id):
        raise ValueError(f"Candidate edge is not in the current OSM graph: {edge_id}")

    u, v, key = edge_id
    osm_attrs = G[u][v][key]
    try:
        edge_length = float(record.get("length_m", osm_attrs.get("length", 0.0)))
    except (TypeError, ValueError, OverflowError):
        edge_length = 0.0

    if edge_length < MIN_PROBE_EDGE_LENGTH_M:
        skipped_short += 1
        continue

    attrs = (
        previous_snapshot[u][v][key]
        if previous_snapshot is not None
        else osm_attrs
    )

    if _has_usable_observation(attrs):
        skipped_observed += 1
        continue

    stored_attempts = attrs.get("traffic_attempt_count")
    if stored_attempts is None:
        stored_attempts = 1 if attrs.get("traffic_directly_queried") is True else 0
    try:
        stored_attempts = max(0, int(stored_attempts))
    except (TypeError, ValueError, OverflowError):
        stored_attempts = 0

    attempts = max(stored_attempts, _edge_history_attempts(edge_id))
    if attempts >= MAX_PROBE_ATTEMPTS_PER_EDGE:
        skipped_max_attempts += 1
        continue

    fraction_index = min(attempts, len(PROBE_FRACTIONS) - 1)
    probe_fraction = float(PROBE_FRACTIONS[fraction_index])

    pending_probe_records.append({
        **record,
        "edge_id": edge_id,
        "length_m": round(edge_length, 2),
        "probe_fraction": probe_fraction,
        "attempt_number": attempts + 1,
        "previous_failure": attrs.get("traffic_last_attempt_reason")
            or attrs.get("traffic_reason"),
    })

selected_probe_records = pending_probe_records[:BATCH_SIZE]
selected_edge_ids = [tuple(record["edge_id"]) for record in selected_probe_records]
selected_probe_fractions = {
    tuple(record["edge_id"]): float(record["probe_fraction"])
    for record in selected_probe_records
}

print("Total candidates:", len(candidate_records))
print("Unique candidates:", len(seen_edge_ids))
print("Already observed and skipped:", skipped_observed)
print("Too-short edges skipped:", skipped_short)
print("Maximum-attempt edges skipped:", skipped_max_attempts)
print("Pending candidates:", len(pending_probe_records))
print("Selected for this run:", len(selected_edge_ids))
print("Maximum requests this run:", BATCH_SIZE)

if selected_probe_records:
    display(pd.DataFrame(selected_probe_records))
else:
    print("No eligible probes remain. The saved Snapshot will be retained; no API call is needed.")

assert len(selected_edge_ids) <= BATCH_SIZE
assert len(selected_edge_ids) == len(set(selected_edge_ids))


Source: Notebook 06 route-focused candidates


NameError: name 'MIN_PROBE_EDGE_LENGTH_M' is not defined

## 4 — Merge helper: preserve older valid observations



In [10]:
_TRAFFIC_VALUE_FIELDS = {
    "current_speed_kph",
    "free_flow_speed_kph",
    "travel_time_s",
    "free_flow_time_s",
    "time_is_imputed",
    "free_flow_is_imputed",
    "fallback_speed_kph",
    "is_closed",
}

def _is_traffic_attribute(name):
    return name.startswith("traffic_") or name in _TRAFFIC_VALUE_FIELDS

def _is_usable_observation(attrs):
    return (
        attrs.get("traffic_matched") is True
        and attrs.get("time_is_imputed") is False
        and attrs.get("traffic_status") in {"red", "yellow", "clear", "closed"}
    )

def merge_traffic_snapshots(base_graph, previous_snapshot, fresh_snapshot, checked_at_utc):
    """Merge one fresh batch while preserving older valid observations."""
    _assert_same_topology(base_graph, fresh_snapshot, "fresh_snapshot")

    if previous_snapshot is None:
        merged = base_graph.copy()
    else:
        _assert_same_topology(base_graph, previous_snapshot, "previous_snapshot")
        merged = previous_snapshot.copy()

    refreshed_edges = 0
    for u, v, key, fresh_attrs in fresh_snapshot.edges(keys=True, data=True):
        was_assessed = (
            bool(fresh_attrs.get("traffic_directly_queried"))
            or bool(fresh_attrs.get("traffic_sampled"))
            or fresh_attrs.get("traffic_match_source") == "reverse_inferred"
        )
        if not was_assessed:
            continue

        target_attrs = merged[u][v][key]
        old_is_usable = _is_usable_observation(target_attrs)
        fresh_is_usable = _is_usable_observation(fresh_attrs)

        old_attempt_count = target_attrs.get("traffic_attempt_count")
        if old_attempt_count is None:
            old_attempt_count = 1 if target_attrs.get("traffic_directly_queried") is True else 0
        try:
            old_attempt_count = max(0, int(old_attempt_count))
        except (TypeError, ValueError, OverflowError):
            old_attempt_count = 0

        if fresh_is_usable or not old_is_usable:
            for attr_name in list(target_attrs):
                if _is_traffic_attribute(attr_name):
                    target_attrs.pop(attr_name, None)
            for attr_name, value in fresh_attrs.items():
                if _is_traffic_attribute(attr_name):
                    target_attrs[attr_name] = copy.deepcopy(value)
        else:
            # Keep the previous speed/time, but preserve all diagnostics from
            # the latest failed attempt separately.
            target_attrs["traffic_last_attempt_reason"] = fresh_attrs.get("traffic_reason")
            target_attrs["traffic_last_attempt_match_reason"] = fresh_attrs.get("traffic_match_reason")
            target_attrs["traffic_last_attempt_midpoint_distance_m"] = fresh_attrs.get("traffic_match_midpoint_distance_m")
            target_attrs["traffic_last_attempt_fraction_near"] = fresh_attrs.get("traffic_match_fraction_near")
            target_attrs["traffic_last_attempt_direction_cosine"] = fresh_attrs.get("traffic_match_direction_cosine")
            target_attrs["traffic_last_probe_fraction"] = fresh_attrs.get("traffic_probe_fraction")
            target_attrs["traffic_last_probe_latitude"] = fresh_attrs.get("traffic_probe_latitude")
            target_attrs["traffic_last_probe_longitude"] = fresh_attrs.get("traffic_probe_longitude")
            target_attrs["traffic_last_attempt_matched"] = bool(fresh_attrs.get("traffic_matched"))

        if fresh_attrs.get("traffic_directly_queried") is True:
            target_attrs["traffic_attempt_count"] = old_attempt_count + 1
        else:
            target_attrs.setdefault("traffic_attempt_count", old_attempt_count)

        target_attrs["traffic_last_checked_at_utc"] = checked_at_utc
        if fresh_is_usable:
            target_attrs["traffic_observed_at_utc"] = checked_at_utc

        refreshed_edges += 1

    merged.graph.update(copy.deepcopy(fresh_snapshot.graph))
    merged.graph["traffic_last_merge_refreshed_edges"] = refreshed_edges
    return merged, refreshed_edges


def count_usable_observations(graph):
    usable = 0
    sampled = 0
    direct = 0
    reverse = 0

    for _, _, _, attrs in graph.edges(keys=True, data=True):
        sampled += bool(attrs.get("traffic_sampled"))
        if _is_usable_observation(attrs):
            usable += 1
            if attrs.get("traffic_match_source") == "direct_probe":
                direct += 1
            elif attrs.get("traffic_match_source") == "reverse_inferred":
                reverse += 1

    total = graph.number_of_edges()
    coverage = 100.0 * usable / total if total else 0.0
    return {
        "usable_observed_edges": usable,
        "direct_observed_edges": direct,
        "reverse_inferred_observed_edges": reverse,
        "sampled_directed_edges": sampled,
        "usable_coverage_percent": round(coverage, 3),
    }

print("PASS: Snapshot merge helpers are ready")


PASS: Snapshot merge helpers are ready


## 5 — Collect live traffic or load the previous snapshot



In [11]:
RUN_LIVE_TRAFFIC = True  # Set False to inspect the saved snapshot without API requests.

fresh_snapshot = None
batch_summary = None
refreshed_edges = 0

if RUN_LIVE_TRAFFIC and selected_edge_ids:
    with TomTomClient() as client:
        fresh_snapshot, batch_summary = collect_traffic_snapshot(
            osm_graph=G,
            client=client,
            max_requests=len(selected_edge_ids),
            selected_edge_ids=selected_edge_ids,
            probe_fractions=selected_probe_fractions,
            pause_s=0.5,
            confidence_threshold=0.65,
        )

    checked_at = batch_summary.get("finished_at_utc") or datetime.now(timezone.utc).isoformat()
    snapshot, refreshed_edges = merge_traffic_snapshots(
        G,
        previous_snapshot,
        fresh_snapshot,
        checked_at,
    )

    summary = dict(batch_summary)
    cumulative = count_usable_observations(snapshot)
    summary["merged_refreshed_edges"] = refreshed_edges
    summary["cumulative_usable_observed_edges"] = cumulative["usable_observed_edges"]
    summary["cumulative_direct_observed_edges"] = cumulative["direct_observed_edges"]
    summary["cumulative_reverse_inferred_observed_edges"] = cumulative["reverse_inferred_observed_edges"]
    summary["cumulative_sampled_directed_edges"] = cumulative["sampled_directed_edges"]
    summary["cumulative_usable_coverage_percent"] = cumulative["usable_coverage_percent"]

    print("========== LIVE COLLECTION COMPLETED ==========")
    print("Requests made:", summary.get("requests_made"))
    print("Request errors:", summary.get("request_errors"))
    print("Newly refreshed directed edges:", refreshed_edges)
    print("Cumulative usable observations:", cumulative["usable_observed_edges"])

else:
    if previous_snapshot is None:
        raise FileNotFoundError(
            "No saved traffic snapshot exists. Select probes and set RUN_LIVE_TRAFFIC=True."
        )
    snapshot = previous_snapshot
    summary = dict(previous_summary)
    if RUN_LIVE_TRAFFIC:
        print("No eligible probe edges. No TomTom request was sent; previous Snapshot retained.")
    else:
        print("Loaded existing snapshot; no API request was made.")

_assert_same_topology(G, snapshot, "working snapshot")
print("Snapshot nodes:", snapshot.number_of_nodes())
print("Snapshot directed edges:", snapshot.number_of_edges())
print("Cumulative observation counts:", count_usable_observations(snapshot))


NameError: name 'selected_edge_ids' is not defined

## 6 — Inspect the selected probes against the merged Snapshot

In [12]:

probe_rows = []

for item in selected_probe_records:
    edge_id = tuple(item["edge_id"])
    u, v, key = edge_id
    attrs = snapshot[u][v][key]

    probe_rows.append({
        "edge_id": edge_id,
        "road_class": item.get("road_class", attrs.get("highway")),
        "sampled": attrs.get("traffic_sampled", False),
        "directly_queried": attrs.get("traffic_directly_queried", False),
        "matched": attrs.get("traffic_matched", False),
        "match_source": attrs.get("traffic_match_source"),
        "match_reason": attrs.get("traffic_match_reason"),
        "traffic_status": attrs.get("traffic_status", "missing"),
        "traffic_reason": attrs.get("traffic_reason"),
        "current_speed_kph": attrs.get("current_speed_kph"),
        "free_flow_speed_kph": attrs.get("free_flow_speed_kph"),
        "confidence": attrs.get("traffic_confidence"),
        "match_distance_m": attrs.get("traffic_match_midpoint_distance_m"),
        "match_overlap": attrs.get("traffic_match_fraction_near"),
        "direction_cosine": attrs.get("traffic_match_direction_cosine"),
        "last_attempt_reason": attrs.get("traffic_last_attempt_reason"),
        "time_is_imputed": attrs.get("time_is_imputed", True),
    })

probe_results_df = pd.DataFrame(probe_rows)
display(probe_results_df)

if RUN_LIVE_TRAFFIC and batch_summary is not None:
    print("Reverse reassignments:",
          batch_summary.get("reverse_direction_reassignments"))
    print("Reverse inferred observations:",
          batch_summary.get("reverse_inferred_observed_edges"))


NameError: name 'selected_probe_records' is not defined

## 7 — Update probe history and audit the Snapshot

In [13]:

if RUN_LIVE_TRAFFIC and batch_summary is not None and batch_summary.get("requests_made", 0) > 0:
    history = update_probe_history(
        fresh_snapshot,
        batch_summary,
        history=history,
    )

    history_path = save_probe_history(history)
    print("Probe history saved:", history_path)
else:
    print("Probe history was not changed because no live collection was run.")

audit = audit_traffic_snapshot(
    G,
    snapshot,
    summary=summary,
)

print("========== TRAFFIC SNAPSHOT AUDIT ==========")
for key, value in audit.items():
    print(f"{key}: {value}")

assert audit["graph_identity_ok"], "Snapshot topology/length audit failed"
assert audit["costs_ok"], "Snapshot contains invalid routing costs"
assert audit["algorithm_smoke_test_ready"], "Snapshot is not structurally ready"

print("PASS: Snapshot passed structural audit")


Probe history was not changed because no live collection was run.


NameError: name 'snapshot' is not defined

## 8 — Build routing graphs for Notebook 06

In [14]:

free_flow_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="free_flow",
)

measured_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="measured",
)

measured_distance = make_routing_graph(
    snapshot,
    objective="distance",
    scenario="measured",
)

print("Free-flow graph edges:", free_flow_time.number_of_edges())
print("Measured-time graph edges:", measured_time.number_of_edges())
print("Measured-distance graph edges:", measured_distance.number_of_edges())

print("PASS: Routing graphs prepared")


NameError: name 'snapshot' is not defined

## 9 — Save the merged Snapshot Bundle



In [15]:

if RUN_LIVE_TRAFFIC and batch_summary is not None and batch_summary.get("requests_made", 0) > 0:
    previous_runs = []
    if previous_bundle is not None:
        previous_runs = list(previous_bundle.get("collection_runs", []))

    previous_runs.append({
        "finished_at_utc": summary.get("finished_at_utc"),
        "requests_made": summary.get("requests_made"),
        "request_errors": summary.get("request_errors"),
        "newly_refreshed_edges": refreshed_edges,
        "cumulative_usable_observed_edges": summary.get(
            "cumulative_usable_observed_edges"
        ),
    })

    traffic_bundle = dict(previous_bundle or {})
    traffic_bundle.update({
        "snapshot": snapshot,
        "summary": summary,
        "plan": selected_probe_records,
        "corridor_plan": plan,
        "selected_edge_ids": selected_edge_ids,
        "collection_runs": previous_runs,
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    })

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    temporary_path = snapshot_path.with_suffix(".tmp")

    with temporary_path.open("wb") as f:
        pickle.dump(
            traffic_bundle,
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    temporary_path.replace(snapshot_path)
    save_summary(summary)

    print("PASS: Merged snapshot bundle saved:", snapshot_path)
    print("Saved usable observations:",
          count_usable_observations(snapshot)["usable_observed_edges"])
    print("Backup remains available:", backup_path if previous_snapshot is not None else "No previous snapshot")

else:
    traffic_bundle = previous_bundle
    print("No new live collection was saved in this run.")
    print("Current snapshot:", snapshot_path)


No new live collection was saved in this run.
Current snapshot: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot.pkl


## 10 — Optional interactive traffic map check

In [16]:

traffic_widget, ui_state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    traffic_graph=snapshot,
)

display(traffic_widget)


NameError: name 'snapshot' is not defined

In [17]:

print("Origin node:", ui_state["origin_node"])
print("Destination node:", ui_state["destination_node"])
print("Last double-click:", ui_state["last_click_latlon"])

assert ui_state["origin_node"] in G
assert ui_state["destination_node"] in G

if ui_state["last_click_latlon"] is None:
    print("Map loaded correctly. No new double-click was made in this session.")
else:
    print("PASS: Interactive map received a double-click.")


NameError: name 'ui_state' is not defined